# Semana 5 — Python III: Clases, Programación Funcional y JSON

**Programación para Ciencia de Datos**<br>
Sesiones: martes 22, miércoles 23 y viernes 25 de septiembre de 2026

> **El miércoles 23 se asigna la Práctica 1.** Y esta semana completa es el corazón de la **Práctica 2**: ahí se te va a pedir explícitamente usar `lambda`, `map()` y `filter()` (sesión 2), y entregar el resultado como un archivo **JSON** (sesión 3).

> **Las clases son opcionales en las prácticas.** Se ven porque te van a servir para organizar código y porque las vas a encontrar en todas las librerías que usaremos después (NumPy, pandas). Pero ninguna práctica te obliga a usarlas.

---

## Sesión 1 — Clases y objetos

### ¿Por qué clases?

Hasta ahora has usado **funciones**: recibes datos, los procesas, devuelves un resultado. Funciona bien, pero mira este problema. En la Práctica 1 terminaste con algo así:

```python
filas = leer_csv("ventas_online-ruido.csv")
cabecera = obtener_cabecera("ventas_online-ruido.csv")
conteo = contar_frecuencias(filas, "producto")
stats = estadisticas(filas, "monto_venta")
vacias = contar_vacias(filas, cabecera)
```

Fíjate que `filas` y `cabecera` se repiten en todas las llamadas. Los datos y las funciones que operan sobre ellos van siempre juntos, pero están separados en el código.

Una **clase** te deja empaquetar los dos en una sola cosa:

```python
datos = Dataset("ventas_online-ruido.csv")
conteo = datos.contar_frecuencias("producto")
stats  = datos.estadisticas("monto_venta")
vacias = datos.contar_vacias()
```

Ya no cargas `filas` ni `cabecera` por todos lados: el objeto los guarda adentro. Eso es todo lo que es una clase — **datos + funciones que trabajan sobre esos datos, en un mismo paquete**.

### Vocabulario mínimo

| Término | Qué es |
|---|---|
| **Clase** | El molde o plantilla. Describe qué datos guarda y qué sabe hacer |
| **Objeto** (o *instancia*) | Algo concreto hecho con ese molde |
| **Atributo** | Un dato que guarda el objeto |
| **Método** | Una función que vive dentro de la clase |
| **`self`** | Dentro de la clase, la forma de referirse al objeto con el que se está trabajando |

La analogía de siempre: la clase es el plano de la casa, el objeto es la casa construida. Con un plano puedes construir muchas casas, y cada una tiene su propio color de pintura.

### Definir y usar una clase

In [ ]:
# La clase mas simple posible
class Estudiante:
    pass    # "pass" significa: por ahora no hay nada adentro

# Crear dos objetos (instancias) de esa clase
a = Estudiante()
b = Estudiante()

print(type(a))
print(a is b)    # son dos objetos distintos, aunque vengan del mismo molde

### `__init__` y `self`

`__init__` es el método que Python llama **automáticamente** cuando creas un objeto. Ahí se definen los atributos iniciales.

`self` es el primer parámetro de todo método, y representa al objeto con el que estás trabajando. Cuando escribes `self.nombre = nombre`, estás diciendo "guarda esto dentro de *este* objeto".

In [ ]:
class Estudiante:
    def __init__(self, nombre, boleta, carrera):
        # self.algo = ... crea un atributo en el objeto
        self.nombre = nombre
        self.boleta = boleta
        self.carrera = carrera

# Al crear el objeto, los argumentos van a __init__ (self se pasa solo)
e1 = Estudiante("Ana Lopez", "2023630123", "LCD")
e2 = Estudiante("Luis Perez", "2023630456", "LCD")

# Leer atributos con punto
print(e1.nombre, "-", e1.boleta)
print(e2.nombre, "-", e2.boleta)

> **¿Por qué `self` se escribe en la definición pero no al llamar?** Porque Python lo pasa por ti. Cuando escribes `e1.saludar()`, Python lo convierte internamente en `Estudiante.saludar(e1)`. Ese `e1` es el `self`. Si olvidas poner `self` como primer parámetro de un método, obtendrás un `TypeError`.

### Cada objeto tiene sus propios atributos

In [ ]:
# Los atributos son independientes entre objetos
e1.carrera = "Ciencia de Datos"

print("e1:", e1.carrera)
print("e2:", e2.carrera)    # e2 no cambio

# Tambien puedes agregar atributos despues de crear el objeto
e1.promedio = 9.2
print("e1 promedio:", e1.promedio)

### Métodos: funciones dentro de la clase

In [ ]:
class Estudiante:
    def __init__(self, nombre, boleta):
        self.nombre = nombre
        self.boleta = boleta
        self.calificaciones = []    # empieza vacia

    def agregar_calificacion(self, valor):
        """Agrega una calificacion a la lista del estudiante."""
        self.calificaciones.append(valor)

    def promedio(self):
        """Calcula el promedio de las calificaciones registradas."""
        if not self.calificaciones:
            return None
        return sum(self.calificaciones) / len(self.calificaciones)

    def aprobado(self):
        """True si el promedio es mayor o igual a 6."""
        p = self.promedio()
        return p is not None and p >= 6


e = Estudiante("Ana Lopez", "2023630123")
e.agregar_calificacion(8.5)
e.agregar_calificacion(9.0)
e.agregar_calificacion(7.5)

print("Calificaciones:", e.calificaciones)
print(f"Promedio: {e.promedio():.2f}")
print("Aprobado:", e.aprobado())

Nota cómo `promedio()` y `aprobado()` no reciben datos: los sacan de `self.calificaciones`. Y `aprobado()` llama a `self.promedio()` — un método puede usar otro de la misma clase.

### Encapsulamiento básico

**Encapsulamiento** es la idea de que el objeto proteja sus datos internos y exponga solo lo necesario. Python no tiene atributos verdaderamente privados; usa una **convención**:

- `self.nombre` → público, úsalo libremente
- `self._cache` → "interno, no lo toques desde fuera" (solo un acuerdo entre programadores, Python no lo impide)

El guion bajo es una señal para quien lea tu código, no un candado.

In [ ]:
class Dataset:
    def __init__(self, filas):
        self.filas = filas          # publico
        self._cache_conteos = {}    # interno: no se espera que lo uses desde fuera

    def n_filas(self):
        return len(self.filas)


d = Dataset([{"producto": "Laptop"}, {"producto": "Mouse"}])
print(d.n_filas())

# Python te DEJA tocar el atributo interno, pero por convencion no deberias
print(d._cache_conteos)

### Métodos especiales: `__str__` y `__len__`

Los métodos con doble guion bajo (`__init__`, `__str__`, `__len__`) le enseñan a tu objeto a comportarse como los tipos de Python. Se les llama *métodos especiales* o *dunder* (*double underscore*).

In [ ]:
class Estudiante:
    def __init__(self, nombre, boleta):
        self.nombre = nombre
        self.boleta = boleta
        self.calificaciones = []

    def __str__(self):
        """Lo que se muestra al hacer print() del objeto."""
        return f"Estudiante({self.nombre}, boleta {self.boleta})"

    def __len__(self):
        """Lo que devuelve len() del objeto."""
        return len(self.calificaciones)


e = Estudiante("Ana Lopez", "2023630123")
e.calificaciones = [8.5, 9.0, 7.5]

print(e)         # usa __str__
print(len(e))    # usa __len__

> **Por qué esto importa para el curso:** cuando escribas `len(mi_array)` en NumPy o `print(mi_dataframe)` en pandas, estarás usando exactamente estos métodos. Las librerías que vamos a ver están construidas con clases; reconocer el patrón te ayuda a leer su documentación.

### Una clase útil para el curso: `ColumnaNumerica`

Vamos a hacer algo que sí vas a poder reutilizar. Esta clase recibe una lista de strings (como salen del CSV), se queda con los que son convertibles a número, y ofrece sus estadísticas.

In [ ]:
class ColumnaNumerica:
    """Envuelve una columna numerica leida como texto desde un CSV."""

    def __init__(self, nombre, valores_texto):
        self.nombre = nombre
        self.valores = self._a_numeros(valores_texto)

    def _a_numeros(self, valores_texto):
        """Convierte a float lo que se pueda; ignora vacios y basura."""
        numeros = []
        for v in valores_texto:
            try:
                numeros.append(float(v))
            except (ValueError, TypeError):
                continue
        return numeros

    def n_validos(self):
        return len(self.valores)

    def minimo(self):
        return min(self.valores) if self.valores else None

    def maximo(self):
        return max(self.valores) if self.valores else None

    def promedio(self):
        if not self.valores:
            return None
        return sum(self.valores) / len(self.valores)

    def resumen(self):
        """Devuelve un diccionario, listo para exportar a JSON."""
        return {
            "columna": self.nombre,
            "n_valores_validos": self.n_validos(),
            "minimo": self.minimo(),
            "maximo": self.maximo(),
            "promedio": self.promedio(),
        }

    def __len__(self):
        return self.n_validos()

    def __str__(self):
        return f"ColumnaNumerica({self.nombre}, {self.n_validos()} validos)"


# Probar con datos sucios, como los del dataset real
crudos = ["15000.50", "350", "", "14500", "abc", "800", "", "5000", "16000"]
col = ColumnaNumerica("monto_venta", crudos)

print(col)
print(col.resumen())

### Ejercicios sesión 1

In [ ]:
# Ejercicio 1: Define una clase llamada Pareja que represente a un equipo del curso.
# En __init__ debe recibir: nombre del equipo, tema y seed, y guardarlos como atributos.
# Agrega un metodo descripcion() que retorne un string como:
#   "Equipo Orugas - tema resenas_cursos - seed 3"

# Tu codigo aqui:

In [ ]:
# Ejercicio 2: Agrega a la clase Pareja un metodo nombre_repo() que retorne el nombre
# que debe tener su repositorio, con el formato: pcd-{tema}-{seed}
# Ejemplo: pcd-resenas_cursos-3

# Tu codigo aqui:

In [ ]:
# Ejercicio 3: Define una clase ColumnaCategorica, parecida a ColumnaNumerica, que:
#   - en __init__ reciba el nombre de la columna y una lista de strings
#   - guarde solo los valores no vacios
#   - tenga un metodo valores_unicos() que retorne un set
#   - tenga un metodo mas_frecuente() que retorne una tupla (valor, conteo)
#   - tenga __len__ que retorne cuantos valores no vacios hay
# Pruebala con: ["Laptop", "Mouse", "Laptop", "", "Teclado", "Mouse", "Laptop", ""]

# Tu codigo aqui:

In [ ]:
# Ejercicio 4: Agrega a ColumnaNumerica un metodo mediana() que retorne la mediana
# de los valores validos (sin usar librerias).
# Recuerda: ordena los valores; si la cantidad es impar, la mediana es el de en medio;
# si es par, es el promedio de los dos centrales.

# Tu codigo aqui:

---

## Sesión 2 — Programación funcional

> **Hoy se asigna la Práctica 1.**

La **programación funcional** es un estilo donde tratas a las funciones como cualquier otro valor: las guardas en variables, las pasas como argumento, las devuelves. En Python esto se usa muchísimo para transformar y filtrar colecciones de datos.

Las tres herramientas de esta sesión:

| Herramienta | Para qué sirve | Cambia el tamaño |
|---|---|---|
| `map(f, datos)` | Aplica `f` a **cada** elemento | No: entran n, salen n |
| `filter(f, datos)` | Deja solo los elementos donde `f` es `True` | Sí: salen n o menos |
| `reduce(f, datos)` | Combina todo en **un solo** valor | Sí: sale 1 |

### `lambda`: funciones de una línea

Una `lambda` es una función anónima (sin nombre) que cabe en una expresión. Estas dos son equivalentes:

```python
def doble(x):
    return x * 2

doble = lambda x: x * 2
```

La sintaxis es `lambda parametros: expresion`. No lleva `return` — el valor de la expresión **es** lo que devuelve. Y solo admite una expresión: si necesitas un `if/else` largo, varias líneas o un bucle, usa `def`.

In [ ]:
# Una lambda guardada en una variable (posible, pero poco idiomatico)
doble = lambda x: x * 2
print(doble(5))

# Lambda con varios parametros
area = lambda base, altura: base * altura / 2
print(area(10, 4))

# Lambda con condicional (sintaxis de expresion: valor_si if cond else valor_no)
clasificar = lambda monto: "alto" if monto > 5000 else "bajo"
print(clasificar(15000))
print(clasificar(350))

> **Cuándo usar `lambda` y cuándo `def`:** usa `lambda` cuando la función es tan corta y tan obvia que ponerle nombre sería ruido, y solo la vas a usar ahí mismo (típicamente como argumento de `map`, `filter` o `sorted`). Si necesitas reutilizarla, documentarla con docstring, o tiene más de una línea de lógica, usa `def`. Guardar una `lambda` en una variable con nombre, como `doble` arriba, es justo el caso donde conviene más un `def`.

### `map()`: transformar cada elemento

`map(funcion, iterable)` aplica la función a cada elemento. **No devuelve una lista**, devuelve un objeto perezoso (*lazy*): no calcula nada hasta que se lo pides. Por eso casi siempre lo envuelves en `list()`.

In [ ]:
precios_texto = ["15000.50", "350", "14500", "800", "5000"]

# map devuelve un objeto map, no una lista
resultado = map(float, precios_texto)
print(resultado)              # <map object at 0x...>
print(list(resultado))        # ahora si se calcula

# Ojo: un objeto map se consume una sola vez
print(list(resultado))        # vacio, ya se agoto

> **Trampa clásica:** `map` y `filter` son de un solo uso. Si recorres el resultado dos veces, la segunda sale vacía. Si vas a usarlo más de una vez, convierte a `list()` desde el principio.

In [ ]:
# map con lambda: aplicar IVA a cada precio
precios = [15000.50, 350.0, 14500.0, 800.0, 5000.0]
con_iva = list(map(lambda p: round(p * 1.16, 2), precios))
print(con_iva)

# map con una funcion con nombre
def normalizar(texto):
    """Quita espacios sobrantes y pasa a minusculas."""
    return texto.strip().lower()

categorias = ["  Laptop ", "MOUSE", "Teclado  ", " monitor"]
print(list(map(normalizar, categorias)))

### `filter()`: quedarte con algunos elementos

In [ ]:
# filter deja pasar los elementos donde la funcion devuelve True
precios = [15000.50, 350.0, 14500.0, 800.0, 5000.0, 320.0]

caros = list(filter(lambda p: p > 5000, precios))
print("Caros:", caros)

# Caso muy comun en este curso: descartar celdas vacias
valores = ["15000.50", "350", "", "14500", "   ", "800"]
no_vacios = list(filter(lambda v: v.strip() != "", valores))
print("No vacios:", no_vacios)

### El orden importa: `filter` y luego `map`

Esta es **la** confusión más frecuente con datos sucios, y la vas a necesitar en la Práctica 2. La columna numérica del dataset trae celdas vacías. Si conviertes antes de filtrar, `float("")` truena.

In [ ]:
valores = ["15000.50", "350", "", "14500", "800"]

# MAL: map primero -> float("") lanza ValueError
try:
    numeros = list(map(float, valores))
except ValueError as e:
    print("Error:", e)

In [ ]:
# BIEN: filter primero (descartar vacios), map despues (convertir)
valores = ["15000.50", "350", "", "14500", "   ", "800"]

limpios = filter(lambda v: v.strip() != "", valores)
numeros = list(map(float, limpios))

print("Numeros:", numeros)
print("Promedio:", sum(numeros) / len(numeros))

> **Regla práctica:** primero `filter` (quitar lo que no sirve), después `map` (transformar lo que quedó). En la Práctica 2 se te pide explícitamente en ese orden.
>
> Cuidado con que `filter(lambda v: v.strip() != "", ...)` solo atrapa los **vacíos**. Si la celda trae texto como `"abc"`, pasa el filtro y después truena en `float()`. Para datos de verdad conviene una función que intente convertir y avise si no pudo — eso lo armamos en la sesión 3.

### `reduce()`: combinar todo en un valor

`reduce` no es una función nativa: hay que importarla de `functools`. Toma los elementos de dos en dos, acumulando un resultado.

Con `reduce(lambda a, b: a + b, [1, 2, 3, 4])` el recorrido es: `1+2=3`, luego `3+3=6`, luego `6+4=10`.

In [ ]:
from functools import reduce

numeros = [15000.50, 350.0, 14500.0, 800.0, 5000.0]

# Suma acumulada
total = reduce(lambda a, b: a + b, numeros)
print("Total:", total)

# Maximo, a mano
maximo = reduce(lambda a, b: a if a > b else b, numeros)
print("Maximo:", maximo)

# Con valor inicial (tercer argumento) - importante si la lista puede estar vacia
total_seguro = reduce(lambda a, b: a + b, [], 0)
print("Total de lista vacia:", total_seguro)

> **En la práctica, casi nunca necesitas `reduce`.** Python ya trae `sum()`, `max()`, `min()`, `len()`, y son más legibles y más rápidas. `reduce` se justifica cuando la combinación es algo que no existe como función nativa. Se ve en el curso porque lo vas a encontrar en código ajeno y porque viene en el examen.

In [ ]:
# Lo mismo que arriba, con funciones nativas: mas corto y mas claro
print("Total:", sum(numeros))
print("Maximo:", max(numeros))

# Un caso donde reduce SI aporta: agrupar en un diccionario
from functools import reduce

categorias = ["Laptop", "Mouse", "Laptop", "Teclado", "Mouse", "Laptop"]

def acumular(conteo, valor):
    conteo[valor] = conteo.get(valor, 0) + 1
    return conteo

print(reduce(acumular, categorias, {}))

### Comprehensions vs `map`/`filter`

Una *list comprehension* hace lo mismo que `map` y `filter`, con otra sintaxis:

```python
[expresion for elemento in iterable if condicion]
#  \_ el map _/                      \_ el filter _/
```

In [ ]:
valores = ["15000.50", "350", "", "14500", "800"]

# Con filter + map
con_fm = list(map(float, filter(lambda v: v.strip() != "", valores)))

# Con comprehension: la misma cosa
con_comp = [float(v) for v in valores if v.strip() != ""]

print(con_fm)
print(con_comp)
print("Iguales:", con_fm == con_comp)

**¿Cuál usar?** En Python la comprehension suele ser la opción más legible, y es lo que verás en la mayoría del código. Pero:

| Situación | Conviene |
|---|---|
| Transformar y/o filtrar, caso normal | **Comprehension** |
| Aplicar una función que ya tiene nombre | `map(float, datos)` lee mejor que `[float(v) for v in datos]` |
| Encadenar varias transformaciones sobre datos enormes | `map`/`filter`, porque son perezosos y no construyen listas intermedias |
| La lógica necesita `if/else`, varias condiciones o anidamiento | **Comprehension** (con `lambda` se vuelve ilegible) |

> **Para la Práctica 2:** el enunciado pide explícitamente al menos un `lambda` y al menos un `map()` o `filter()`. Úsalos ahí. Fuera de ese requisito, escoge lo que se lea mejor.

### Bonus: `sorted()` con `key`

`sorted()` acepta una `lambda` en el parámetro `key` para decidir por qué criterio ordenar. Esto sí lo vas a usar seguido.

In [ ]:
conteo = {"Laptop": 12543, "Mouse": 8021, "Teclado": 13021, "Monitor": 4500}

# Ordenar los pares (clave, valor) por el valor, de mayor a menor
ranking = sorted(conteo.items(), key=lambda par: par[1], reverse=True)
for producto, veces in ranking:
    print(f"{producto:10s} {veces}")

print()
# El mas frecuente, sin ordenar todo
print("Mas frecuente:", max(conteo, key=conteo.get))

### Ejercicios sesión 2

In [ ]:
# Ejercicio 5: Dada la lista de abajo, usa map y lambda para obtener una nueva lista
# con la longitud de cada string.

palabras = ["Laptop", "Mouse", "Teclado", "Monitor", "Impresora"]

# Tu codigo aqui:

In [ ]:
# Ejercicio 6: Dada la lista de abajo (valores de una columna numerica leida del CSV),
# usa filter y map EN ESE ORDEN para obtener una lista de floats sin vacios.
# Luego calcula e imprime el promedio.

crudos = ["850.50", "1200", "", "920.30", "1500", "   ", "750", ""]

# Tu codigo aqui:

In [ ]:
# Ejercicio 7: Escribe la misma transformacion del ejercicio 6 usando una
# list comprehension en una sola linea. Verifica que el resultado sea identico.

# Tu codigo aqui:

In [ ]:
# Ejercicio 8: Usa filter con lambda para quedarte con los correos VALIDOS de la lista.
# Considera valido un correo que contenga exactamente un "@" y al menos un "." despues del "@".

correos = ["ana@gmail.com", "carlosgmail.com", "diego@@live.com",
           "val@outlook.com", "miguel@", "sofia@yahoo.com"]

# Tu codigo aqui:

In [ ]:
# Ejercicio 9: Usa reduce para encontrar el string MAS LARGO de la lista de palabras
# del ejercicio 5. (Pista: la lambda recibe dos strings y devuelve el mas largo.)

# Tu codigo aqui:

---

## Sesión 3 — JSON y ejercicio integrador

Dos cosas hoy. Primero **JSON**, el formato en el que vas a entregar los resultados de la Práctica 2. Después juntamos las piezas del semestre hasta aquí — **funciones**, **clases** y **programación funcional** — para procesar un dataset como el de tus prácticas y guardar la salida en JSON.

El plan: leer un CSV separado por `|`, convertirlo en una lista de diccionarios, calcular sobre él lo mismo que pide la Práctica 2, y escribir el resultado como `.json`.

> **Recuerda el separador:** el archivo termina en `.csv` pero las columnas van separadas por `|`, no por comas. Es a propósito: las columnas de dirección y de texto libre traen comas en su propio contenido, así que un `.split(",")` las partiría mal.

### JSON: qué es y para qué lo usamos

**JSON** (*JavaScript Object Notation*) es un formato de texto para guardar datos estructurados. Un CSV es una tabla plana: filas y columnas, y nada más. JSON permite **anidar**: diccionarios dentro de diccionarios, listas dentro de diccionarios. Por eso la Práctica 2 pide JSON y no CSV — su salida tiene secciones adentro de secciones.

Si abres un `.json` con un editor de texto, verás algo casi idéntico a un diccionario de Python:

```json
{
  "archivo": "ventas_online-ruido.csv",
  "seed": 3,
  "estadisticas_numerica_1": {
    "columna": "monto_venta",
    "minimo": 50.0,
    "maximo": 15000.0
  }
}
```

El módulo `json` viene en la **biblioteca estándar**: no se instala con `pip`, solo se importa. Por eso la Práctica 2 lo permite aunque prohíba librerías externas.

### Correspondencia de tipos Python ↔ JSON

JSON no tiene todos los tipos de Python. Al guardar, cada tipo se traduce:

| Python | JSON | Al leer de vuelta regresa como |
|---|---|---|
| `dict` | objeto `{}` | `dict` |
| `list`, `tuple` | arreglo `[]` | `list` (¡una tupla regresa como lista!) |
| `str` | string | `str` |
| `int`, `float` | número | `int` / `float` |
| `True` / `False` | `true` / `false` | `bool` |
| `None` | `null` | `None` |
| **`set`** | **no existe** | **lanza `TypeError`** |

Las dos últimas filas son las que causan problemas. Ponles atención.

### Escribir: `json.dump` y `json.dumps`

In [ ]:
import json

resumen = {
    "archivo": "ventas_ejemplo.csv",
    "seed": 3,
    "n_filas": 12,
    "columnas": ["producto", "monto_venta", "email"],
    "tiene_vacios": True,
    "promedio": None,
}

# json.dumps  -> devuelve un STRING (fijate en la "s" de string)
texto = json.dumps(resumen)
print(type(texto))
print(texto)

> **`dump` vs `dumps`:** `json.dump(objeto, archivo)` escribe directo a un archivo. `json.dumps(objeto)` devuelve un string. La `s` final es de *string*. Lo mismo del otro lado: `json.load(archivo)` lee de un archivo, `json.loads(string)` lee de un string. Confundirlas es un error muy común.

In [ ]:
# Sin indent, todo sale en una sola linea ilegible
print("SIN indent:")
print(json.dumps(resumen))
print()

# Con indent=2 queda formateado y legible
print("CON indent=2:")
print(json.dumps(resumen, indent=2))

### `ensure_ascii=False`: los acentos

Por defecto `json` escapa todo lo que no sea ASCII, y los acentos quedan ilegibles. `ensure_ascii=False` los deja tal cual. La Práctica 2 lo pide explícitamente.

In [ ]:
con_acentos = {"tema": "citas médicas", "columna": "especialidad",
               "nota": "El análisis incluye categorías con ñ y acentuación"}

print("Por defecto (ensure_ascii=True):")
print(json.dumps(con_acentos, indent=2))
print()
print("Con ensure_ascii=False:")
print(json.dumps(con_acentos, indent=2, ensure_ascii=False))

### Escribir a un archivo

In [ ]:
# La forma que vas a usar en la practica
with open("resumen_demo.json", "w", encoding="utf-8") as f:
    json.dump(con_acentos, f, indent=2, ensure_ascii=False)

# Verificar que quedo bien escrito
print(open("resumen_demo.json", encoding="utf-8").read())

> **El `encoding="utf-8"` al abrir el archivo no es opcional.** En Windows, `open()` usa por defecto una codificación que no maneja acentos, y el archivo queda corrupto o truena. Pon `encoding="utf-8"` siempre, tanto al escribir como al leer.

### Leer: `json.load` y `json.loads`

Esto lo vas a necesitar en la **Práctica 4**, donde tendrás que abrir el `analisis_python.json` que generaste en la P2 y comparar su promedio contra el que calcules con pandas.

In [ ]:
# Leer de un archivo
with open("resumen_demo.json", "r", encoding="utf-8") as f:
    recuperado = json.load(f)

print(type(recuperado))
print(recuperado["tema"])

# Leer de un string
desde_texto = json.loads('{"a": 1, "b": [2, 3]}')
print(desde_texto, type(desde_texto))

### Trampa 1: los `set` no se pueden guardar en JSON

Esto te va a pasar en la Práctica 2. La función `valores_unicos()` devuelve un `set` (así lo pide el enunciado), pero el JSON de salida necesita un **arreglo** en `valores_unicos_categorica.valores`. Si pasas el set directo, truena.

In [ ]:
# Esto es un set: lo que devuelve valores_unicos() en la Practica 2
unicos = {"Laptop", "Mouse", "Teclado", "Monitor", "Impresora"}

# MAL: json no sabe que hacer con un set
try:
    json.dumps({"valores": unicos})
except TypeError as e:
    print("TypeError:", e)

In [ ]:
# BIEN: convertirlo antes de guardar. Hay dos formas, y NO dan lo mismo:
print("con sorted():", json.dumps(sorted(unicos), ensure_ascii=False))
print("con list()  :", json.dumps(list(unicos), ensure_ascii=False))

> **Usa `sorted()`, no `list()`.** Vuelve a ejecutar la celda de arriba **varias veces** y fíjate: la línea de `sorted()` sale siempre igual, la de `list()` cambia de orden en cada ejecución. El orden de un `set` en Python no está garantizado entre ejecuciones, así que con `list()` tu archivo de salida cambiaría sin que tú hayas cambiado nada — y eso hace imposible comparar dos corridas o revisar un diff en Git. Con `sorted()` el resultado es reproducible.

### Trampa 2: las llaves de un objeto JSON siempre son strings

Si tu diccionario tiene llaves numéricas, JSON las convierte a texto — y al leer de vuelta **ya no vuelven a ser números**.

In [ ]:
por_anio = {2023: 1500, 2024: 2300}

guardado = json.dumps(por_anio)
print("Guardado:", guardado)          # las llaves salen entre comillas

recuperado = json.loads(guardado)
print("Recuperado:", recuperado)
print("Tipo de la llave:", type(list(recuperado.keys())[0]))   # str, no int

No es un problema para la Práctica 2 (ahí las llaves del `conteo_por_categoria` son nombres de categoría, o sea strings de por sí), pero conviene saberlo para cuando agrupes por año o por número.

> **Una última, por si la encuentras:** `float("nan")` y `float("inf")` se escriben como `NaN` e `Infinity`, que **no son JSON válido** según el estándar. Python los lee de vuelta sin chistar, pero otras herramientas los rechazan. Si una estadística te sale `nan`, conviértela a `None` antes de guardar.

### Ejercicios de JSON

In [ ]:
# Ejercicio 10: Crea un diccionario con tus datos de equipo (nombre, tema, seed,
# integrantes como lista) y guardalo en un archivo "mi_equipo.json" con
# indent=2 y ensure_ascii=False. Luego vuelve a leerlo con json.load
# y verifica que el contenido sea igual al original.

# Tu codigo aqui:

In [ ]:
# Ejercicio 11: El siguiente diccionario NO se puede guardar como JSON tal cual.
# Encuentra por que y arreglalo para que json.dumps funcione.

problematico = {
    "columna": "producto",
    "valores_unicos": {"Laptop", "Mouse", "Teclado"},
    "n_filas": 12,
}

# Tu codigo aqui:

---

### Ahora el integrador

In [ ]:
%%writefile ventas_ejemplo.csv
producto|monto_venta|minutos_en_sitio|email|comentario_cliente
Laptop|15000.50|45.0|ana@gmail.com|Excelente equipo, llego a tiempo
Mouse|350|12.0|luis@hotmail.com|Bueno
Laptop|14500||carlosgmail.com|Caro pero vale la pena
Teclado|800|8.5||Sin comentarios
Monitor|5000|33.0|sofia@yahoo.com|
Laptop||51.0|diego@live.com|Muy satisfecho con la compra
Mouse|400|5.0|val@outlook.com|Cumple
Impresora|3500|27.5|andrea@gmail.com|Tarda en imprimir, Calle Juarez #12, CDMX
Mouse|320||miguel@gmail.com|Regular
Laptop|16000|62.0|paula@gmail.com|La recomiendo
Teclado|750|9.0|jose@gmail.com|Buena relacion precio-calidad
Monitor|abc|41.0|rosa@gmail.com|Dato capturado mal

### Paso 1 — Leer el CSV como lista de diccionarios

En la Práctica 1 trabajaste con listas de listas y accedías por índice (`fila[2]`). Eso se rompe en cuanto cambia el orden de las columnas — y en el dataset real **el orden de las columnas es aleatorio**. Con diccionarios accedes por nombre (`fila["monto_venta"]`), que es a prueba de eso.

In [ ]:
def leer_csv(ruta, separador="|"):
    """Lee un CSV y lo devuelve como lista de diccionarios.
    Cada fila es un dict {nombre_columna: valor}."""
    with open(ruta, "r", encoding="utf-8") as f:
        lineas = f.read().strip().split("\n")

    cabecera = lineas[0].split(separador)
    datos = []
    for linea in lineas[1:]:
        valores = linea.split(separador)
        datos.append(dict(zip(cabecera, valores)))
    return datos


datos = leer_csv("ventas_ejemplo.csv")

print("Filas:", len(datos))
print("Columnas:", list(datos[0].keys()))
print()
print("Primera fila:")
for clave, valor in datos[0].items():
    print(f"  {clave}: {valor!r}")

> **`zip(cabecera, valores)`** empareja cada nombre de columna con su valor: `[("producto", "Laptop"), ("monto_venta", "15000.50"), ...]`. Y `dict()` convierte esa lista de pares en un diccionario. Es el idioma estándar para esto en Python.

### Paso 2 — Conversión segura a número

Ya vimos que `filter` + `map` con `v.strip() != ""` no alcanza: la fila de `Monitor` trae `monto_venta` igual a `"abc"`, que pasa el filtro de vacíos y después truena en `float()`.

La solución es una función que intente convertir y devuelva `None` si no puede. Así el filtro decide sobre el resultado de la conversión, no sobre el texto.

In [ ]:
def a_float(texto):
    """Intenta convertir a float. Devuelve None si no se puede."""
    try:
        return float(texto)
    except (ValueError, TypeError):
        return None


# Probar con los tres casos que trae el dataset
for v in ["15000.50", "", "abc", "   ", "350"]:
    print(f"{v!r:12s} -> {a_float(v)}")

In [ ]:
# Ahora si: map para convertir, filter para quedarse con lo convertible.
# Aqui el orden se invierte respecto a la sesion 2, porque el filtro
# actua sobre el RESULTADO de la conversion, no sobre el texto crudo.
def columna_numerica(datos, columna):
    """Extrae una columna como lista de floats, descartando lo no convertible."""
    crudos = map(lambda fila: fila.get(columna, ""), datos)
    convertidos = map(a_float, crudos)
    return list(filter(lambda x: x is not None, convertidos))


montos = columna_numerica(datos, "monto_venta")
print("Validos:", len(montos), "de", len(datos), "filas")
print(montos)

### Paso 3 — Las funciones que pide la Práctica 2

Cuatro funciones, cada una con una responsabilidad clara.

In [ ]:
def valores_unicos(datos, columna):
    """Set de valores distintos de una columna, sin contar vacios."""
    valores = map(lambda fila: fila.get(columna, "").strip(), datos)
    return set(filter(lambda v: v != "", valores))


def conteo_por_categoria(datos, columna):
    """Diccionario {valor: cuantas veces aparece}, sin contar vacios."""
    conteo = {}
    for fila in datos:
        valor = fila.get(columna, "").strip()
        if valor == "":
            continue
        conteo[valor] = conteo.get(valor, 0) + 1
    return conteo


def estadisticas(datos, columna):
    """min, max, promedio y n_validos de una columna numerica."""
    numeros = columna_numerica(datos, columna)
    if not numeros:
        return {"columna": columna, "n_valores_validos": 0,
                "minimo": None, "maximo": None, "promedio": None}
    return {
        "columna": columna,
        "n_valores_validos": len(numeros),
        "minimo": min(numeros),
        "maximo": max(numeros),
        "promedio": round(sum(numeros) / len(numeros), 2),
    }


def percentil(numeros, p):
    """Percentil aproximado por posicion, como lo pide la Practica 2."""
    if not numeros:
        return None
    ordenados = sorted(numeros)
    pos = int(p * len(ordenados))
    pos = min(pos, len(ordenados) - 1)    # no pasarse del final
    return ordenados[pos]


print("Unicos en 'producto':", valores_unicos(datos, "producto"))
print()
print("Conteo:", conteo_por_categoria(datos, "producto"))
print()
print("Stats de 'monto_venta':", estadisticas(datos, "monto_venta"))

### Paso 4 — Filtrado con `filter` y `lambda`

Primero una advertencia sobre el percentil, porque con pocos datos se comporta de forma que parece un error.

In [ ]:
montos = columna_numerica(datos, "monto_venta")
print("Valores validos:", len(montos))
print("Ordenados:", sorted(montos))
print()

# Con 10 valores, int(0.9 * 10) = 9, que es la ULTIMA posicion de la lista:
# el percentil 90 coincide con el maximo, y nada lo supera.
umbral_90 = percentil(montos, 0.9)
print("Percentil 90:", umbral_90, "(es el maximo!)")
print("Registros que lo superan:", len([m for m in montos if m > umbral_90]))

> **Esto no es un bug de la fórmula, es un efecto de la muestra chica.** `int(0.9 * n)` cae en la última posición cuando `n` es pequeño. En tu dataset real hay ~98,000 valores válidos, así que `int(0.9 * 98000) = 88200` queda muy adentro de la lista y el filtro sí deja pasar alrededor del 10% de los registros. La fórmula es la que pide la Práctica 2; úsala tal cual.
>
> Para que el ejemplo de aquí se vea, bajamos el percentil a 75.

In [ ]:
# Con la muestra chica usamos el percentil 75 para que el filtro se note
umbral = percentil(montos, 0.75)
print("Umbral (percentil 75):", umbral)

# Filtrar los registros que lo superan
caros = list(filter(lambda fila: (a_float(fila.get("monto_venta", "")) or 0) > umbral,
                   datos))

print("Registros que cumplen:", len(caros))

# Porcentaje SOBRE LOS VALORES VALIDOS de la columna, no sobre el total de filas
porcentaje = round(100 * len(caros) / len(montos), 2)
print(f"Porcentaje: {porcentaje}%")
print()
for fila in caros:
    print(" ", fila["producto"], "-", fila["monto_venta"])

> **Ojo con el denominador.** El porcentaje va sobre el número de valores **válidos** de la columna numérica, no sobre el total de filas del archivo. Son distintos: aquí hay 12 filas pero solo 10 montos convertibles. Es un detalle que la Práctica 2 pide explícitamente.

Para comprobar que la fórmula del percentil 90 sí se comporta bien cuando hay muchos datos, generemos una lista grande:

In [ ]:
# Simular una columna numerica con 10,000 valores (sin librerias externas)
import random
random.seed(42)
muchos = [round(random.uniform(100, 20000), 2) for _ in range(10000)]

u90 = percentil(muchos, 0.9)
arriba = len(list(filter(lambda x: x > u90, muchos)))

print("n =", len(muchos))
print("Percentil 90:", u90)
print("Valores que lo superan:", arriba)
print(f"Porcentaje: {round(100 * arriba / len(muchos), 2)}%    <- cerca del 10%, como se espera")

### Paso 5 — Empaquetar todo en una clase

Las funciones ya hacen el trabajo. Una clase las junta con los datos, para no andar pasando `datos` en cada llamada. Esto es **opcional** en la práctica, pero compara qué tan distinto se lee el código de abajo.

In [ ]:
class Dataset:
    """Un dataset del curso, leido desde un CSV separado por '|'."""

    def __init__(self, ruta, separador="|"):
        self.ruta = ruta
        self.filas = leer_csv(ruta, separador)
        self.columnas = list(self.filas[0].keys()) if self.filas else []

    # --- acceso a columnas ---

    def columna(self, nombre):
        """La columna como lista de strings, tal cual viene del archivo."""
        return [fila.get(nombre, "") for fila in self.filas]

    def numerica(self, nombre):
        """La columna como lista de floats, descartando lo no convertible."""
        return columna_numerica(self.filas, nombre)

    # --- analisis ---

    def valores_unicos(self, columna):
        return valores_unicos(self.filas, columna)

    def conteo(self, columna):
        return conteo_por_categoria(self.filas, columna)

    def mas_frecuente(self, columna):
        c = self.conteo(columna)
        if not c:
            return None, 0
        valor = max(c, key=c.get)
        return valor, c[valor]

    def estadisticas(self, columna):
        return estadisticas(self.filas, columna)

    def contar_vacias(self):
        """Celdas vacias por columna."""
        return {col: sum(1 for fila in self.filas
                         if fila.get(col, "").strip() == "")
                for col in self.columnas}

    def filtrar(self, condicion):
        """Filas que cumplen una condicion. 'condicion' es una funcion
        que recibe una fila (dict) y devuelve True o False."""
        return list(filter(condicion, self.filas))

    # --- metodos especiales ---

    def __len__(self):
        return len(self.filas)

    def __str__(self):
        return f"Dataset({self.ruta}: {len(self.filas)} filas, {len(self.columnas)} columnas)"


d = Dataset("ventas_ejemplo.csv")
print(d)
print()
print("Mas frecuente en 'producto':", d.mas_frecuente("producto"))
print("Stats de 'minutos_en_sitio':", d.estadisticas("minutos_en_sitio"))
print()
print("Celdas vacias por columna:")
for col, n in d.contar_vacias().items():
    print(f"  {col:20s} {n}")

In [ ]:
# El metodo filtrar() acepta cualquier lambda: la clase no tiene que saber
# de antemano por que criterios vas a querer filtrar.
laptops = d.filtrar(lambda fila: fila["producto"] == "Laptop")
print("Laptops:", len(laptops))

con_email_valido = d.filtrar(lambda fila: fila["email"].count("@") == 1)
print("Con email valido:", len(con_email_valido))

largos = d.filtrar(lambda fila: len(fila["comentario_cliente"]) > 20)
print("Con comentario largo:", len(largos))

### Paso 6 — Exportar a JSON

Ya tenemos todo calculado en diccionarios, y el módulo `json` ya lo conoces del inicio de la sesión. Juntarlo es casi inmediato: así se ve la salida completa que pide la Práctica 2.

In [ ]:
import json

mas_frecuente, veces = d.mas_frecuente("producto")
montos = d.numerica("monto_venta")

# En tu practica esto va con percentil 0.9; aqui usamos 0.75 solo porque
# la muestra de ejemplo es muy chica (ver la advertencia del paso 4).
umbral = percentil(montos, 0.75)
caros = d.filtrar(lambda fila: (a_float(fila.get("monto_venta", "")) or 0) > umbral)

resultado = {
    "archivo": "ventas_ejemplo.csv",
    "pareja": "Nombre A y Nombre B",
    "seed": 3,
    "valores_unicos_categorica": {
        "columna": "producto",
        "n_valores_unicos": len(d.valores_unicos("producto")),
        "valores": sorted(d.valores_unicos("producto")),
    },
    "conteo_por_categoria": d.conteo("producto"),
    "categoria_mas_frecuente": {"valor": mas_frecuente, "conteo": veces},
    "estadisticas_numerica_1": d.estadisticas("monto_venta"),
    "estadisticas_numerica_2": d.estadisticas("minutos_en_sitio"),
    "filtrado": {
        "condicion": "monto_venta > " + str(umbral),
        "umbral_usado": umbral,
        "n_registros_que_cumplen": len(caros),
        "porcentaje": round(100 * len(caros) / len(montos), 2),
    },
}

with open("analisis_ejemplo.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)

print("Archivo escrito. Contenido:")
print()
print(json.dumps(resultado, indent=2, ensure_ascii=False))

> **`ensure_ascii=False`** es importante: sin él, `json.dump` escribe los acentos como `\u00f3` y el archivo queda ilegible. Y `indent=2` lo deja formateado en varias líneas en lugar de una sola línea gigante.

### Ejercicios sesión 3

Estos son prácticamente la Práctica 2 en versión chica. Resuélvelos con el CSV de ejemplo y después te será directo aplicarlo a tu dataset real.

In [ ]:
# Ejercicio 12: Agrega a la clase Dataset un metodo mediana(columna) que devuelva
# la mediana de una columna numerica. Reutiliza el metodo numerica().

# Tu codigo aqui:

In [ ]:
# Ejercicio 13: Agrega a Dataset un metodo agrupar_por(categorica, numerica) que
# devuelva un diccionario {categoria: promedio de la columna numerica}.
# Ejemplo esperado con el CSV de ejemplo:
#   {"Laptop": 15166.83, "Mouse": 356.67, ...}
# Pista: recorre las filas acumulando los valores de cada categoria en una lista,
# y al final calcula el promedio de cada lista.

# Tu codigo aqui:

In [ ]:
# Ejercicio 14: Usando map, filter y lambda (sin bucles for), calcula cuantos
# registros tienen un comentario_cliente que mencione una direccion
# (pista: contienen una coma dentro del comentario).

# Tu codigo aqui:

In [ ]:
# Ejercicio 15: Escribe una funcion resumen_completo(dataset, categorica, numerica)
# que reciba un objeto Dataset y devuelva un diccionario con:
#   - n_filas
#   - n_columnas
#   - valores unicos de la categorica
#   - la categoria mas frecuente con su conteo
#   - estadisticas de la numerica
#   - celdas vacias por columna
# Guardalo como JSON en un archivo llamado "resumen_ejercicio.json".

# Tu codigo aqui:

In [ ]:
# Ejercicio 16 (reto): Define una clase ColumnaTexto para las columnas de texto libre.
# Debe tener:
#   - __init__(nombre, valores): guarda solo los no vacios
#   - longitud_promedio(): promedio de caracteres
#   - palabra_mas_comun(): la palabra que mas aparece en todos los textos juntos
#     (pista: junta todo con " ".join(), separa con .split(), y cuenta)
#   - __len__
# Pruebala con la columna comentario_cliente del CSV de ejemplo.

# Tu codigo aqui:

---

## Resumen de la semana

**Clases**

- `class Nombre:` define el molde; `Nombre()` crea un objeto
- `__init__(self, ...)` corre al crear el objeto y define sus atributos
- `self` es el objeto actual; va como primer parámetro de todo método, pero no se pasa al llamar
- `_atributo` señala "interno" por convención, no por obligación
- `__str__` y `__len__` le enseñan a tu objeto a responder a `print()` y `len()`
- Sirven para empaquetar datos + funciones que operan sobre ellos. **Opcionales en las prácticas**

**Programación funcional**

- `lambda params: expresion` — función de una línea, sin `return`
- `map(f, datos)` transforma cada elemento; `filter(f, datos)` selecciona; `reduce(f, datos)` combina en uno
- `map` y `filter` son perezosos y de un solo uso: envuélvelos en `list()`
- `reduce` se importa de `functools`, y casi siempre `sum`/`max`/`min` son mejor opción
- Las comprehensions hacen lo mismo y suelen leerse mejor

**JSON**

- `json` es de la biblioteca estándar: se importa, no se instala
- `json.dump(obj, archivo)` escribe a archivo; `json.dumps(obj)` devuelve un string (la `s` es de *string*)
- `json.load(archivo)` y `json.loads(string)` leen de vuelta
- Siempre: `indent=2` para que sea legible, `ensure_ascii=False` para los acentos, y `encoding="utf-8"` al abrir el archivo
- Un `set` **no** se puede guardar: conviértelo con `sorted()` (mejor que `list()`, porque el resultado es reproducible)
- Las llaves de un objeto JSON siempre regresan como `str`, y una tupla regresa como lista

**Lo que más se equivoca**

1. **El orden `filter` → `map`** con datos sucios. `float("")` truena, así que primero descarta y luego convierte.
2. Filtrar solo por vacíos no basta: `"abc"` también revienta `float()`. Usa una función tipo `a_float` que devuelva `None`.
3. Recorrer dos veces un `map`/`filter`: la segunda vez sale vacío.
4. Olvidar `self` como primer parámetro de un método.
5. El denominador del porcentaje: va sobre los **valores válidos** de la columna, no sobre el total de filas.
6. Pasarle un `set` a `json.dumps` y toparse con `TypeError`.
7. Confundir `dump` con `dumps` (o `load` con `loads`).

**Para la Práctica 2** (se asigna el mar 6-oct): se piden al menos 4 funciones, al menos un `lambda` y al menos un `map()` o `filter()`. Las clases son opcionales. Salida en JSON con `json.dump(..., indent=2, ensure_ascii=False)`.

> **Y para la Práctica 4:** ahí vas a tener que **leer** con `json.load` el `analisis_python.json` que entregues en la P2, para comparar su promedio contra el que calcules con pandas. Así que conviene que tu JSON de la P2 quede bien hecho — lo vas a volver a usar.